# Stage 6 — Expanded Layer Causal Feature Freeze R5

**Notebook 12 · atomic claim-cleanup recovery repair R5**

CPU High-RAM is preferred because this notebook performs host-CPU causal feature engineering, not model-heavy training. CPU, GPU, and TPU sessions remain portable, and no accelerator use is claimed. R5 preserves all R4 causal/scientific and crash-recovery gates and atomically retires completed JSON claims before resumable tombstone cleanup. Fresh targets and action timing remain excluded.

## Operator settings

In [ ]:
import os
REQUESTED_WORKERS = "auto"  # @param {type:"string"}
ACCELERATOR_ENVIRONMENT = "CPU"  # @param ["CPU", "GPU", "TPU"]
REQUESTED_WORKERS = os.environ.get("STAGE6_REQUESTED_WORKERS", REQUESTED_WORKERS).strip()
if REQUESTED_WORKERS != "auto":
    if int(REQUESTED_WORKERS) < 1:
        raise ValueError("REQUESTED_WORKERS must be auto or a positive integer")
print({"requested_workers": REQUESTED_WORKERS, "accelerator_environment": ACCELERATOR_ENVIRONMENT, "preferred_host": "CPU High-RAM", "execution": "host_cpu_process_work", "accelerator_acceleration_claimed": False})


## Mount Drive and verify the exact R5 package

In [ ]:
import hashlib
import importlib.metadata
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path, PurePosixPath

for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[variable] = "1"

PINNED_RUNTIME_DISTRIBUTIONS = {"catboost": "1.2.10", "joblib": "1.5.2", "lightgbm": "4.6.0", "numpy": "2.2.6", "pandas": "2.3.3", "psutil": "7.0.0", "pyarrow": "21.0.0", "scikit-learn": "1.7.2", "threadpoolctl": "3.6.0", "xgboost": "3.3.0"}
PINNED_RUNTIME_MODULES = {"catboost": "catboost", "joblib": "joblib", "lightgbm": "lightgbm", "numpy": "numpy", "pandas": "pandas", "psutil": "psutil", "pyarrow": "pyarrow", "scikit-learn": "sklearn", "threadpoolctl": "threadpoolctl", "xgboost": "xgboost"}
mismatched = {
    distribution: version
    for distribution, version in PINNED_RUNTIME_DISTRIBUTIONS.items()
    if importlib.util.find_spec(PINNED_RUNTIME_MODULES[distribution]) is None
    or importlib.metadata.version(distribution) != version
}
preloaded = [PINNED_RUNTIME_MODULES[name] for name in mismatched if PINNED_RUNTIME_MODULES[name] in sys.modules]
if mismatched:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *[f"{name}=={version}" for name, version in PINNED_RUNTIME_DISTRIBUTIONS.items()]])
if preloaded:
    raise RuntimeError("PINNED_ENVIRONMENT_RESTART_REQUIRED: restart and Run all; " + ",".join(preloaded))

try:
    from google.colab import drive
except ImportError as error:
    raise RuntimeError("This release notebook is intended for Google Colab") from error
drive.mount("/content/drive")
STAGE6_DRIVE_ROOT = Path("/content/drive/MyDrive/Poker Assistant Model/Stage 6 - Qualified Layers Improvement")
PACKAGE_PATH = STAGE6_DRIVE_ROOT / "inputs" / "POKERNOW_ASSISTANT_STAGE6_EXPANDED_FEATURE_FREEZE_INPUTS_R5.zip"
EXPECTED_PACKAGE_SHA256 = "3c47557099eedee0e958e2b95edc2b131e91128db6a7e2820e2a3f6ff1ab240f"
if hashlib.sha256(PACKAGE_PATH.read_bytes()).hexdigest() != EXPECTED_PACKAGE_SHA256:
    raise RuntimeError("R5 expanded feature package SHA-256 mismatch")
PACKAGE_ROOT = Path("/content/stage6_expanded_feature_freeze_r5")
if PACKAGE_ROOT.exists():
    shutil.rmtree(PACKAGE_ROOT)
PACKAGE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(PACKAGE_PATH) as archive:
    names = archive.namelist()
    if not names or names[-1] != "PACKAGE_MANIFEST.json" or len(names) != len(set(names)):
        raise RuntimeError("invalid R5 package membership")
    for name in names:
        pure = PurePosixPath(name)
        if pure.is_absolute() or ".." in pure.parts or "\\" in name:
            raise RuntimeError(f"unsafe package member: {name}")
    archive.extractall(PACKAGE_ROOT)
package_manifest = json.loads((PACKAGE_ROOT / "PACKAGE_MANIFEST.json").read_text(encoding="utf-8"))
for member in package_manifest["members"]:
    raw = (PACKAGE_ROOT / member["path"]).read_bytes()
    if len(raw) != member["bytes"] or hashlib.sha256(raw).hexdigest() != member["sha256"]:
        raise RuntimeError(f"R5 package member identity mismatch: {member['path']}")
sys.path.insert(0, str(PACKAGE_ROOT / "src"))
from stage6_expanded_downstream_rebind_v2 import activate_downstream_rebind_v2
R2_DOWNSTREAM_REBIND = activate_downstream_rebind_v2()
print(json.dumps({"event": "stage6_expanded_feature_r5_ready", "package_sha256": EXPECTED_PACKAGE_SHA256, "tree_history_execution": "host_cpu_process_work", "tpu_acceleration_claimed": False}, sort_keys=True))


## Run exact-equivalence preflight and the R2 feature freeze

In [ ]:
from stage6_expanded_feature_freeze_v2 import FEATURE_RUN_ID_V2, run_feature_freeze_v2
result = run_feature_freeze_v2(STAGE6_DRIVE_ROOT, REQUESTED_WORKERS, ACCELERATOR_ENVIRONMENT)
RUN_IDENTITY = FEATURE_RUN_ID_V2
PRIMARY_MANIFEST_PATH = Path(result["manifest_path"])
result


## Publish and download the compact returned result

In [ ]:
from stage6_expanded_notebook_controller import publish_compact_result
returned_result = publish_compact_result(
    STAGE6_DRIVE_ROOT, token="12-r5", run_identity=RUN_IDENTITY,
    result=result, primary_manifest_path=PRIMARY_MANIFEST_PATH,
)
print(returned_result)
try:
    from google.colab import files
    files.download(returned_result["path"])
except Exception as error:
    print({"download_status": "manual_download_may_be_required", "detail": str(error), "path": returned_result["path"]})
